### 1. Convert the XLSX file to CSV format.

In [ ]:
import pandas as pd

XLSX_PATH = "../Data/CAS_KPBMA_MAP3K5_IC50s.xlsx"       # original xlsx file 
SHEET_INDEX = 1
OUT_CSV = "../Data/CAS_KPBMA_MAP3K5_IC50s_2nd_sheet.csv"       # output csv file

df = pd.read_excel(XLSX_PATH, sheet_name=SHEET_INDEX, header=1, engine="openpyxl")

df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved to {OUT_CSV}")

Saved to ../Data/CAS_KPBMA_MAP3K5_IC50s_2nd_sheet.csv


### 2. Analyze the converted CAS CSV data by species (Human, Homo sapiens, Mouse).

In [14]:
import numpy as np 
CAS_DF = pd.read_csv('../Data/CAS_KPBMA_MAP3K5_IC50s_2nd_sheet.csv')

# Convert IC50 from micromolar (uM) to nanomolar (nM)
CAS_DF['IC50_nM'] = CAS_DF['Single Value (Parsed)'] * 1000

# Convert nM IC50 values to pIC50
# pIC50 = 9 - log10(IC50_nM)
CAS_DF['pIC50'] = 9 - np.log10(CAS_DF['IC50_nM'])

CAS_DF.head()


,Substance Name,Source Type,Document Identifier,SMILES,Assay Name,Measurement Function,Assay Parameter,Display Measurement,Measurement Prefix (Parsed),Single Value (Parsed),Low-End Value (Parsed),High-End Value (Parsed),Measurement Unit,pX Value,Cell Line,Species,Disease,IC50_nM,pIC50
0,5′-ATP,journal,"Chem. Biol., 2011, 18 (6), 699-710",O[C@H]1[C@H](N2C=3C(N=C2)=C(N)N=CN3)O[C@H](COP...,NaN,Inhibitor,IC50,2.43333 µM,NaN,2.43333,NaN,NaN,µM,5.61380,HL-60,HOMO SAPIENS,NaN,2433.33,5.613799
1,6-Methylthioguanine,journal,"Bioorg. Med. Chem., 2011, 19 (1), 486-489",S(C)C1=C2C(=NC(N)=N1)N=CN2,Mobility shift assay,NaN,IC50,13.3 µM,NaN,13.30000,NaN,NaN,µM,4.87615,NaN,NaN,NaN,13300.00,4.876148
2,N-Phenyl-9H-purin-6-amine,journal,"Bioorg. Med. Chem. Lett., 2018, 28 (3), 400-404",N(C1=C2C(N=CN2)=NC=N1)C3=CC=CC=C3,AlphaScreen assay,Inhibitor,IC50,> 10 µM,>,10.00000,NaN,NaN,µM,5.00000,NaN,NaN,NaN,10000.00,5.000000
3,6-(1-Piperidinyl)-9H-purine,journal,"Bioorg. Med. Chem., 2011, 19 (1), 486-489",C12=C(N=CN=C1N=CN2)N3CCCCC3,Mobility shift assay,NaN,IC50,17 µM,NaN,17.00000,NaN,NaN,µM,4.76955,NaN,NaN,NaN,17000.00,4.769551
4,"1-Acetyl-3H-naphtho[1,2,3-de]quinoline-2,7-dione",journal,"J. Med. Chem., 2011, 54 (8), 2680-2686",C(C)(=O)C1=C2C=3C(C(=O)C=4C2=CC=CC4)=CC=CC3NC1=O,NaN,NaN,IC50,15 µM,NaN,15.00000,NaN,NaN,µM,4.82391,NaN,HOMO SAPIENS,NaN,15000.00,4.823909


In [15]:
species_data = {}

for species_name in ["HOMO SAPIENS", "Human", "Mouse"]:
    species_mask = CAS_DF["Species"].str.upper() == species_name.upper()
    species_df = CAS_DF[species_mask]
    
    # Select rows where pIC50 is not NaN
    species_df_filtered = species_df[species_df["pIC50"].notna()]
    
    if len(species_df_filtered) > 0:
        species_data[species_name] = {
            "count": len(species_df_filtered),
            "pIC50_min": species_df_filtered["pIC50"].min(),
            "pIC50_max": species_df_filtered["pIC50"].max(),
            "pIC50_mean": species_df_filtered["pIC50"].mean()
        }


# Count NaN species entries only where pIC50 is available
nan_count = CAS_DF[CAS_DF["pIC50"].notna()]["Species"].isna().sum()

print("Species counts and pIC50 ranges:")
for species, data in species_data.items():
    print(f"  {species}: {data['count']} entries")
    print(f"    pIC50 range: {data['pIC50_min']:.6f} ~ {data['pIC50_max']:.6f}")
    print(f"    pIC50 mean: {data['pIC50_mean']:.6f}")
    print()

print(f"  NaN species (where pIC50 exists): {nan_count} entries")

# Overall pIC50 range (excluding NaN)
single_value_min = CAS_DF["pIC50"].min()
single_value_max = CAS_DF["pIC50"].max()
print(f"\nOverall pIC50 range: {single_value_min:.6f} ~ {single_value_max:.6f}")

Species counts and pIC50 ranges:
  HOMO SAPIENS: 335 entries
    pIC50 range: 4.200659 ~ 9.397940
    pIC50 mean: 6.783605

  Human: 532 entries
    pIC50 range: 6.000000 ~ 13.000000
    pIC50 mean: 7.421284

  Mouse: 2 entries
    pIC50 range: 4.829738 ~ 4.974694
    pIC50 mean: 4.902216

  NaN species (where pIC50 exists): 2178 entries

Overall pIC50 range: 3.301030 ~ 13.000000


### 3. Apply filtering after CAS data analysis.

In [16]:
Clean_CAS_df = CAS_DF.copy()

# Filter rows where species is HOMO SAPIENS or Human
species_mask = (
    Clean_CAS_df['Species'].str.upper().isin(['HOMO SAPIENS', 'HUMAN']) | 
    Clean_CAS_df['Species'].isna()
)

filtered_cas_df = Clean_CAS_df[species_mask].copy()

# Keep only rows with valid pIC50 values
cas_final_df = filtered_cas_df[filtered_cas_df['pIC50'].notna()][['SMILES', 'pIC50']].copy()

# Rename columns
cas_final_df = cas_final_df.rename(columns={
    'SMILES': 'Smiles'
})

# Quick check
print("Filtering results:")
print(f"  Original rows: {len(Clean_CAS_df)}")
print(f"  Rows after species filter (HOMO SAPIENS + Human): {len(filtered_cas_df)}")
print(f"  Rows with valid pIC50: {len(cas_final_df)}")

print("\nSpecies distribution:")
species_counts = filtered_cas_df['Species'].value_counts()
for species, count in species_counts.items():
    print(f"  {species}: {count} entries")

print("\nFinal data sample:")
print(cas_final_df.head(10))

# pIC50 summary statistics
print(f"\npIC50 statistics:")
print(f"  Min: {cas_final_df['pIC50'].min():.4f}")
print(f"  Max: {cas_final_df['pIC50'].max():.4f}")
print(f"  Mean: {cas_final_df['pIC50'].mean():.4f}")

# Save to CSV
cas_final_df.to_csv('../Data/CAS_clean_SP_pIC50.csv', index=False)



Filtering results:
  Original rows: 3452
  Rows after species filter (HOMO SAPIENS + Human): 3450
  Rows with valid pIC50: 3045

Species distribution:
  Human: 658 entries
  HOMO SAPIENS: 509 entries

Final data sample:
                                               Smiles     pIC50
0   O[C@H]1[C@H](N2C=3C(N=C2)=C(N)N=CN3)O[C@H](COP...  5.613799
1                          S(C)C1=C2C(=NC(N)=N1)N=CN2  4.876148
2                   N(C1=C2C(N=CN2)=NC=N1)C3=CC=CC=C3  5.000000
3                         C12=C(N=CN=C1N=CN2)N3CCCCC3  4.769551
4    C(C)(=O)C1=C2C=3C(C(=O)C=4C2=CC=CC4)=CC=CC3NC1=O  4.823909
5               N(C1=C2C(N=CN2)=NC=N1)C3=CC(Cl)=CC=C3  5.000000
6                      N(CCC)(CCC)C1=C2C(N=CN2)=NC=N1  4.588380
7                 N(C1=NC2=C(C=N1)C=CC=C2)C3=CC=CC=C3  5.000000
10  C[C@]12N3C=4C5=C(C6=C(C4C=7C3=CC=CC7)CNC6=O)C=...  7.612610
11  C[C@]12N3C=4C5=C(C6=C(C4C=7C3=CC=CC7)CNC6=O)C=...  7.642065

pIC50 statistics:
  Min: 3.3010
  Max: 13.0000
  Mean: 7.2742


### 4. Analyze the provided ChEMBL data.

In [17]:
Chembl_df = pd.read_csv('../Data/ChEMBL_ASK1(IC50).csv', sep=';')
# Check pChEMBL counts and NaN counts by BAO Label
bao_counts = {}

# Compute total and NaN counts for each BAO Label
bao_labels = ['single protein format', 'assay format', 'cell-based format']

for label in bao_labels:
    mask = Chembl_df['BAO Label'] == label
    label_df = Chembl_df[mask]
    
    total_count = label_df.shape[0]
    nan_count = label_df['pChEMBL Value'].isna().sum()
    valid_count = total_count - nan_count
    
    bao_counts[label] = {
        'total': total_count,
        'nan': nan_count,
        'valid': valid_count
    }

# Print results
print("Counts and pChEMBL NaN values by BAO Label:")
for label, data in bao_counts.items():
    print(f"  {label}:")
    print(f"    Total: {data['total']}")
    print(f"    pChEMBL NaN: {data['nan']}")
    print(f"    Valid pChEMBL: {data['valid']}")
    print()

# Overall counts
total_count = Chembl_df.shape[0]
total_nan = Chembl_df['pChEMBL Value'].isna().sum()
total_valid = total_count - total_nan

print("Overall dataset:")
print(f"  Total: {total_count}")
print(f"  pChEMBL NaN: {total_nan}")
print(f"  Valid pChEMBL: {total_valid}")

# NaN ratio
print(f"\nOverall pChEMBL NaN ratio: {(total_nan/total_count)*100:.1f}%")
#Chembl_single_protein_df.to_csv('/home/sechoi/Projects/DACON/Summary/Chembl_single_protein_df.csv', index=False)

Counts and pChEMBL NaN values by BAO Label:
  single protein format:
    Total: 414
    pChEMBL NaN: 61
    Valid pChEMBL: 353

  assay format:
    Total: 50
    pChEMBL NaN: 34
    Valid pChEMBL: 16

  cell-based format:
    Total: 360
    pChEMBL NaN: 15
    Valid pChEMBL: 345

Overall dataset:
  Total: 824
  pChEMBL NaN: 110
  Valid pChEMBL: 714

Overall pChEMBL NaN ratio: 13.3%


### 5. Apply filtering after ChEMBL data analysis.

In [18]:
# Exclude assay format and keep only single protein format and cell-based format
format_mask = Chembl_df['BAO Label'].isin(['single protein format', 'cell-based format'])
chembl_filtered_df = Chembl_df[format_mask].copy()

# Remove rows with NaN pChEMBL values
chembl_clean_df = chembl_filtered_df[chembl_filtered_df['pChEMBL Value'].notna()].copy()

chembl_final_df = chembl_clean_df[['Smiles', 'pChEMBL Value']].copy()
chembl_final_df = chembl_final_df.rename(columns={'pChEMBL Value': 'pIC50'})

# Quick check
print(f"Original rows: {len(Chembl_df)}")
print(f"Rows after BAO Label filter: {len(chembl_filtered_df)}")
print(f"Rows after removing NaN pChEMBL: {len(chembl_clean_df)}")
print(f"Final rows: {len(chembl_final_df)}")

# BAO Label distribution
print("\nBAO Label distribution:")
bao_counts = chembl_filtered_df['BAO Label'].value_counts()
for label, count in bao_counts.items():
    print(f"  {label}: {count} entries")

print("\nFinal data sample:")
print(chembl_final_df.head())

print(f"\npIC50 range: {chembl_final_df['pIC50'].min():.4f} ~ {chembl_final_df['pIC50'].max():.4f}")
print(f"pIC50 mean: {chembl_final_df['pIC50'].mean():.4f}")

# Save to CSV if needed
chembl_final_df.to_csv('../Data/chembl_clean_noassay_pIC50.csv', index=False)

Original rows: 824
Rows after BAO Label filter: 774
Rows after removing NaN pChEMBL: 698
Final rows: 698

BAO Label distribution:
  single protein format: 414 entries
  cell-based format: 360 entries

Final data sample:
                                              Smiles  pIC50
0   Cn1cc(Cl)c2cnc(NC(=O)c3ccc([C@](C)(O)CO)cc3)cc21   7.42
1         Cc1cc2c(-c3ccc(S(=O)(=O)NCCN)s3)ccnc2[nH]1   6.60
2  Cc1cc(C)c2nc(N3C(=O)C(O)=C(C(=O)c4ccc(Cl)cc4)C...   5.20
3  Cc1ccc(C(=O)C2=C(O)C(=O)N(c3nc4ccc(F)cc4s3)C2c...   5.12
4  CCOc1ccc2nc(N3C(=O)C(O)=C(C(=O)c4ccccc4)C3c3cc...   5.38

pIC50 range: 4.0000 ~ 9.1600
pIC50 mean: 6.8315


### 6. Analyze the provided PubChem data.

In [19]:
Pubchem_df = pd.read_csv('../Data/Pubchem_ASK1.csv', sep=',')

# Check Activity_Value ranges and NaN counts by Activity
activity_counts = {}

# Compute counts, Activity_Value ranges, and NaN counts for each Activity value
activity_values = ['Active', 'Inactive', 'Inconclusive', 'Unspecified']

for activity in activity_values:
    mask = Pubchem_df['Activity'] == activity
    activity_df = Pubchem_df[mask]
    
    total_count = activity_df.shape[0]
    nan_count = activity_df['Activity_Value'].isna().sum()
    valid_count = total_count - nan_count
    
    # Compute Activity_Value range (excluding NaN)
    if valid_count > 0:
        min_val = activity_df['Activity_Value'].min()
        max_val = activity_df['Activity_Value'].max()
        mean_val = activity_df['Activity_Value'].mean()
    else:
        min_val = max_val = mean_val = None
    
    activity_counts[activity] = {
        'total': total_count,
        'nan': nan_count,
        'valid': valid_count,
        'min': min_val,
        'max': max_val,
        'mean': mean_val
    }

# Print results
print("Activity counts and Activity_Value summary:")
for activity, data in activity_counts.items():
    print(f"  {activity}:")
    print(f"    Total: {data['total']}")
    print(f"    Activity_Value NaN: {data['nan']}")
    print(f"    Valid Activity_Value: {data['valid']}")
    
    if data['valid'] > 0:
        print(f"    Activity_Value range: {data['min']:.6f} ~ {data['max']:.6f}")
        print(f"    Activity_Value mean: {data['mean']:.6f}")
    else:
        print("    Activity_Value range: no data")
        print("    Activity_Value mean: no data")
    print()

# Overall counts
total_count = Pubchem_df.shape[0]
total_nan = Pubchem_df['Activity_Value'].isna().sum()
total_valid = total_count - total_nan

print("Overall dataset:")
print(f"  Total: {total_count}")
print(f"  Activity_Value NaN: {total_nan}")
print(f"  Valid Activity_Value: {total_valid}")

# NaN ratio
print(f"\nOverall Activity_Value NaN ratio: {(total_nan/total_count)*100:.1f}%")


Activity counts and Activity_Value summary:
  Active:
    Total: 953
    Activity_Value NaN: 274
    Valid Activity_Value: 679
    Activity_Value range: 0.000100 ~ 10.000000
    Activity_Value mean: 1.609882

  Inactive:
    Total: 22071
    Activity_Value NaN: 21730
    Valid Activity_Value: 341
    Activity_Value range: 10.218000 ~ 27.499000
    Activity_Value mean: 18.398062

  Inconclusive:
    Total: 4
    Activity_Value NaN: 4
    Valid Activity_Value: 0
    Activity_Value range: no data
    Activity_Value mean: no data

  Unspecified:
    Total: 767
    Activity_Value NaN: 639
    Valid Activity_Value: 128
    Activity_Value range: 0.100000 ~ 500.000000
    Activity_Value mean: 20.612758

Overall dataset:
  Total: 23795
  Activity_Value NaN: 22647
  Valid Activity_Value: 1148

Overall Activity_Value NaN ratio: 95.2%


/tmp/ipykernel_2063018/1262468187.py:1: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  Pubchem_df = pd.read_csv('../Data/Pubchem_ASK1.csv', sep=',')


### 7. Apply filtering after PubChem data analysis.

In [20]:
# Condition 1: rows where Activity is Active and Activity_Type is IC50
condition_active = (Pubchem_df['Activity'] == 'Active') & (Pubchem_df['Activity_Type'] == 'IC50')
active_ic50_df = Pubchem_df[condition_active].copy()

# Condition 2: rows where Activity is Unspecified, Activity_Type is IC50, and Activity_Value < 10
condition_unspecified = (
    (Pubchem_df['Activity'] == 'Unspecified') & 
    (Pubchem_df['Activity_Type'] == 'IC50') & 
    (Pubchem_df['Activity_Value'] < 10)
)
unspecified_ic50_df = Pubchem_df[condition_unspecified].copy()

# Combine rows from both conditions
filtered_pubchem_df = pd.concat([active_ic50_df, unspecified_ic50_df], ignore_index=True)

# Quick check
print("Filtering results:")
print(f"  Active + IC50: {len(active_ic50_df)}")
print(f"  Unspecified + IC50 + Activity_Value < 10: {len(unspecified_ic50_df)}")
print(f"  Total filtered rows: {len(filtered_pubchem_df)}")

print(f"\nOriginal rows: {len(Pubchem_df)}")
print(f"Rows after filtering: {len(filtered_pubchem_df)}")

# Activity distribution in filtered data
print("\nActivity distribution in filtered rows:")
activity_dist = filtered_pubchem_df['Activity'].value_counts()
for activity, count in activity_dist.items():
    print(f"  {activity}: {count} entries")

# Select required columns and rename
pubchem_final_df = filtered_pubchem_df[['SMILES', 'Activity_Value']].copy()

# Rename column names
pubchem_final_df = pubchem_final_df.rename(columns={'SMILES': 'Smiles'})

# Convert Activity_Value to pIC50
# Activity_Value * 1000 (uM -> nM) -> 9 - log10(IC50_nM)
pubchem_final_df['pIC50'] = 9 - np.log10(pubchem_final_df['Activity_Value'] * 1000)

# Quick check
print("Final data schema:")
print(f"  Columns: {list(pubchem_final_df.columns)}")
print(f"  Row count: {len(pubchem_final_df)}")

print("\nFinal data sample:")
print(pubchem_final_df.head(10))

# pIC50 summary statistics
print(f"\npIC50 statistics:")
print(f"  Min: {pubchem_final_df['pIC50'].min():.4f}")
print(f"  Max: {pubchem_final_df['pIC50'].max():.4f}")
print(f"  Mean: {pubchem_final_df['pIC50'].mean():.4f}")
print(f"  Std: {pubchem_final_df['pIC50'].std():.4f}")

# Save to CSV
pubchem_final_df[['Smiles', 'pIC50']].to_csv('../Data/Pubchem_clean_pIC50.csv', index=False)


Filtering results:
  Active + IC50: 564
  Unspecified + IC50 + Activity_Value < 10: 31
  Total filtered rows: 595

Original rows: 23795
Rows after filtering: 595

Activity distribution in filtered rows:
  Active: 564 entries
  Unspecified: 31 entries
Final data schema:
  Columns: ['Smiles', 'Activity_Value', 'pIC50']
  Row count: 595

Final data sample:
                                              Smiles  Activity_Value  \
0  CC1=CC(=C(C=C1S(=O)(=O)N)C(=O)NC2=CC=CC(=N2)C3...         0.00010   
1  CCS(=O)(=O)NC1=CC(=C(C=C1)OC)C(=O)NC2=CC=CC(=N...         0.00032   
2  CNC(=O)C1CCN(CC1)C2=CN=C(C=C2N3C=C(N=C3)C4CC4)...         0.00040   
3  CC1=CC(=C(C=C1C(=O)N)C(=O)NC2=CC=CC(=N2)C3=NN=...         0.00042   
4  CC12C(C(CC(O1)N3C4=CC=CC=C4C5=C6C(=C7C8=CC=CC=...         0.00069   
5  CC(C(F)(F)F)N1C=NN=C1C2=NC(=CS2)NC(=O)C3=NC=C(...         0.00070   
6  C1CC1C2=CN(C=N2)C3=CC(=NC=C3N4CC(C4)C#N)C(=O)N...         0.00090   
7  CC(C)N1C=NN=C1C2=NC(=CC=C2)NC(=O)C3=C(C=CC(=C3...         0.00100

### 8. Preprocess SMILES data and merge dataframes.

In [21]:
from rdkit import Chem
import re

def remove_cl_salts(smiles):
    """
    Remove salt fragments such as .cl, .cl.cl, .Cl, and .Cl.Cl from SMILES.
    """
    if pd.isna(smiles) or smiles is None:
        return None
    
    # Remove .cl, .cl.cl, .Cl, .Cl.Cl patterns (case-insensitive)
    cleaned_smiles = re.sub(r'\.cl\.cl\.?', '', smiles, flags=re.IGNORECASE)  # .cl.cl or .cl.cl.
    cleaned_smiles = re.sub(r'\.cl\.?', '', cleaned_smiles, flags=re.IGNORECASE)  # .cl or .cl.
    
    # Additional cleanup for .Cl and .Cl.Cl variants
    cleaned_smiles = re.sub(r'\.Cl\.Cl\.?', '', cleaned_smiles)  # .Cl.Cl or .Cl.Cl.
    cleaned_smiles = re.sub(r'\.Cl\.?', '', cleaned_smiles)  # .Cl or .Cl.
    
    # Remove repeated dots (e.g., .. -> .)
    cleaned_smiles = re.sub(r'\.+', '.', cleaned_smiles)
    
    # Trim leading/trailing dots
    cleaned_smiles = cleaned_smiles.strip('.')
    
    return cleaned_smiles if cleaned_smiles else None

def clean_and_canonicalize(smiles):
    """
    Remove .cl salts and canonicalize SMILES.
    """
    # Step 1: remove .cl salts
    cleaned_smiles = remove_cl_salts(smiles)
    
    if cleaned_smiles is None:
        return None
    
    # Step 2: canonicalize
    try:
        mol = Chem.MolFromSmiles(cleaned_smiles)
        if mol is None:
            return None
        
        # Keep only the largest fragment (remove counter-ions)
        frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
        if not frags:
            return None
            
        largest_frag = max(frags, key=lambda m: m.GetNumAtoms())

        # Convert to canonical SMILES
        return Chem.MolToSmiles(largest_frag, canonical=True)
    
    except:
        return None

# Apply to CAS data
cleaned_cas_df = pd.read_csv('../Data/CAS_clean_SP_pIC50.csv')
cleaned_chembl_df = pd.read_csv('../Data/chembl_clean_noassay_pIC50.csv')
cleaned_pubchem_df = pd.read_csv('../Data/Pubchem_clean_pIC50.csv')
add_chembl_df = pd.read_csv('../Data/additional_chembl_ask1_data.csv')

# Add Canonical_Smiles to each dataframe
cleaned_cas_df['Canonical_Smiles'] = cleaned_cas_df['Smiles'].apply(clean_and_canonicalize)
cleaned_chembl_df['Canonical_Smiles'] = cleaned_chembl_df['Smiles'].apply(clean_and_canonicalize)
cleaned_pubchem_df['Canonical_Smiles'] = cleaned_pubchem_df['Smiles'].apply(clean_and_canonicalize)
add_chembl_df['Canonical_Smiles'] = add_chembl_df['Smiles'].apply(clean_and_canonicalize)
add_chembl_df['pIC50'] = -np.log10(add_chembl_df['Standard_Value']*1e-9)

# Build final merged dataframe
train_df = pd.concat([cleaned_cas_df, cleaned_chembl_df, cleaned_pubchem_df, add_chembl_df], ignore_index=True)
train_df_final = train_df[['Canonical_Smiles', 'pIC50']].copy()
train_df_final.to_csv('../Data/train_df.csv', index=False)

### 10. For duplicated Canonical SMILES entries, split and save datasets by max or min pIC50.

In [ ]:
# Remove rows with NaN pIC50
train_df_clean = train_df_final.dropna(subset=['pIC50']).copy()

print(f"Row count after dropping NaN pIC50: {len(train_df_clean)}")

# Check duplicates by Canonical_Smiles
duplicate_counts = train_df_clean['Canonical_Smiles'].value_counts()
duplicate_smiles = duplicate_counts[duplicate_counts > 1]

print(f"\nNumber of duplicated SMILES: {len(duplicate_smiles)}")

# For duplicated SMILES, compute max and min pIC50
max_pic50_df = train_df_clean.groupby('Canonical_Smiles')['pIC50'].max().reset_index()
min_pic50_df = train_df_clean.groupby('Canonical_Smiles')['pIC50'].min().reset_index()

# Rename columns
max_pic50_df = max_pic50_df.rename(columns={'pIC50': 'pIC50'})
min_pic50_df = min_pic50_df.rename(columns={'pIC50': 'pIC50'})

# Save CSV files
max_pic50_df.to_csv('../Data/train_df_final_max.csv', index=False)
min_pic50_df.to_csv('../Data/train_df_final_min.csv', index=False)

Row count after dropping NaN pIC50: 4366

Number of duplicated SMILES: 830
